# DREAMT Sleep Staging — WatchSleepNet Task Ablations

**Paper:** Wang et al., *WatchSleepNet: A Novel Model and Pretraining Approach for Advancing Sleep Staging with Smartwatches*, 2025.  
https://doi.org/10.48550/arXiv.2501.17268

**Dataset:** DREAMT (PhysioNet) — https://physionet.org/content/dreamt/

This notebook demonstrates the `SleepStagingDREAMT` task on the DREAMT wearable dataset and includes three novel ablation studies **not present in the original paper**:

| Ablation | Variable | Paper default | What we test |
|---|---|---|---|
| 1 — Label granularity | `num_classes` | 3 (Wake/NREM/REM) | 3-class vs 4-class (N1/N2/N3 split) |
| 2 — Accelerometer | `use_accelerometer` | False (IBI only) | IBI-only vs IBI + ACC_X/Y/Z |
| 3 — Epoch duration | `epoch_seconds` | 30 s | 15 s / 30 s / 60 s |

**Quick start (no download required):** Set `USE_DEMO = True` below.  
**Real data:** Set `USE_DEMO = False` and point `DREAMT_ROOT` at your local DREAMT 2.1.0 directory.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
USE_DEMO    = True          # True  → synthetic data (no download needed)
                            # False → set DREAMT_ROOT to your local path
DREAMT_ROOT = "/path/to/dreamt/2.1.0"

In [ ]:
import collections
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

## Demo mode: synthetic DREAMT directory

When `USE_DEMO = True` we build a minimal DREAMT directory in a temp folder so the notebook is fully self-contained. Each synthetic patient has 60 epochs of 30 s (3 840 rows at 64 Hz) with stages cycling through W / N1 / N2 / N3 / R.

In [ ]:
_demo_tmpdir = None

def _build_demo_root(n_patients: int = 6, n_rows: int = 3840) -> str:
    """Create a minimal synthetic DREAMT directory tree."""
    global _demo_tmpdir
    _demo_tmpdir = tempfile.mkdtemp(prefix="dreamt_demo_")
    root = Path(_demo_tmpdir)
    (root / "data_64Hz").mkdir()
    (root / "data_100Hz").mkdir()

    rng = np.random.default_rng(0)
    stage_cycle = (
        ["W"] * 640 + ["N1"] * 640 + ["N2"] * 640 + ["N3"] * 640 + ["R"] * 640
    ) * 2  # 5 × 640 = 3 200 rows; repeated so n_rows=3 840 is covered

    rows = []
    for i in range(1, n_patients + 1):
        sid = f"S{i:03d}"

        ibi = np.zeros(n_rows)
        beat_idx = np.arange(0, n_rows, 51)  # ~1 beat per 0.8 s
        ibi[beat_idx] = rng.uniform(0.7, 1.1, len(beat_idx))

        df = pd.DataFrame({
            "TIMESTAMP": np.arange(n_rows) / 64.0,
            "BVP":       rng.standard_normal(n_rows),
            "HR":        rng.integers(50, 90, n_rows).astype(float),
            "EDA":       rng.uniform(0.0, 1.0, n_rows),
            "TEMP":      rng.uniform(33.0, 37.0, n_rows),
            "ACC_X":     rng.standard_normal(n_rows),
            "ACC_Y":     rng.standard_normal(n_rows),
            "ACC_Z":     rng.standard_normal(n_rows),
            "IBI":       ibi,
            "Sleep_Stage": stage_cycle[:n_rows],
        })
        df.to_csv(root / "data_64Hz" / f"{sid}_whole_df.csv", index=False)
        pd.DataFrame({"a": [1]}).to_csv(
            root / "data_100Hz" / f"{sid}_PSG_df.csv", index=False
        )

        rows.append({
            "SID": sid, "AGE": rng.integers(25, 65),
            "GENDER": rng.choice(["M", "F"]), "BMI": rng.integers(18, 40),
            "OAHI": rng.integers(0, 30), "AHI": rng.integers(0, 30),
            "Mean_SaO2": f"{rng.integers(90, 99)}%",
            "Arousal Index": rng.integers(5, 30),
            "MEDICAL_HISTORY": "None", "Sleep_Disorders": "None",
        })

    pd.DataFrame(rows).to_csv(root / "participant_info.csv", index=False)
    print(f"[demo] Synthetic DREAMT root: {root}")
    return str(root)


root = _build_demo_root() if USE_DEMO else DREAMT_ROOT
print(f"Using root: {root}")

## Step 1 — Load DREAMTDataset

In [ ]:
from pyhealth.datasets import DREAMTDataset

dreamt = DREAMTDataset(root=root)
dreamt.stats()
print(f"Patients loaded: {len(dreamt.unique_patient_ids)}")

In [ ]:
from pyhealth.tasks import SleepStagingDREAMT

def summarise(task_ds, name: str) -> None:
    """Print epoch count and class distribution for a task dataset."""
    n = len(task_ds.samples)
    counts = dict(sorted(collections.Counter(s["label"] for s in task_ds.samples).items()))
    print(f"  [{name}]")
    print(f"    Total epochs : {n}")
    print(f"    Label dist   : {counts}")

---
## Ablation 1 — Label Granularity: 3-class vs 4-class

The paper uses **3-class** staging (Wake / NREM / REM), merging N1, N2, and N3 into a single NREM class.  
We test whether separating NREM into its constituent stages (**4-class**: Wake / N1 / N2 / N3 / REM) improves clinical granularity, at the cost of a harder classification problem.

**Hypothesis:** finer labels give a model more signal to differentiate NREM depth but may hurt overall accuracy due to inter-stage similarity.

In [ ]:
task_3cls = SleepStagingDREAMT(num_classes=3)
task_4cls = SleepStagingDREAMT(num_classes=4)

ds_3cls = dreamt.set_task(task_3cls)
ds_4cls = dreamt.set_task(task_4cls)

print("Label granularity comparison:")
summarise(ds_3cls, "3-class  Wake=0 / NREM=1 / REM=2")
summarise(ds_4cls, "4-class  Wake=0 / N1=1 / N2=2 / N3=3 / REM=4")

print(
    "\nObservation: both datasets share the same epoch count; "
    "4-class spreads NREM epochs across three labels."
)

---
## Ablation 2 — Accelerometer Augmentation: IBI-only vs IBI + ACC

The paper uses only **IBI** (Inter-Beat Interval) as the model input.  
We test whether adding raw wrist **accelerometer** signals (ACC_X / ACC_Y / ACC_Z) improves **Wake detection**, since physical movement is a strong wakefulness indicator.

**Hypothesis:** ACC data captures motion patterns invisible to cardiac signals, boosting Wake F1 without hurting NREM/REM classification.

In [ ]:
task_ibi_only = SleepStagingDREAMT(num_classes=3, use_accelerometer=False)
task_ibi_acc  = SleepStagingDREAMT(num_classes=3, use_accelerometer=True)

ds_ibi_only = dreamt.set_task(task_ibi_only)
ds_ibi_acc  = dreamt.set_task(task_ibi_acc)

print("Accelerometer augmentation comparison:")
summarise(ds_ibi_only, "IBI-only        input keys: ibi_sequence")
summarise(ds_ibi_acc,  "IBI + ACC       input keys: ibi_sequence, accelerometer")

if ds_ibi_acc.samples:
    acc_shape = ds_ibi_acc.samples[0]["accelerometer"].shape
    print(f"\nACC tensor shape per epoch: {acc_shape}  (rows × 3 axes)")

print(
    "\nTo train: replace feature_keys=['ibi_sequence'] with "
    "['ibi_sequence', 'accelerometer'] and compare Wake F1."
)

---
## Ablation 3 — Epoch Duration: 15 s / 30 s / 60 s

The paper fixes each epoch at **30 seconds** (the PSG standard).  
We test shorter (15 s) and longer (60 s) windows to explore the tradeoff between temporal resolution and per-epoch IBI context.

**Hypothesis:** shorter windows increase epoch count and temporal resolution but give the model fewer heartbeats per sample; longer windows provide richer IBI context but may blur stage transitions.

In [ ]:
print(f"{'Epoch (s)':<10} {'Total epochs':<15} {'Avg IBI vals/epoch':<20}")
print("-" * 45)

for epoch_secs in (15, 30, 60):
    task_ep = SleepStagingDREAMT(epoch_seconds=epoch_secs, num_classes=3)
    ds_ep   = dreamt.set_task(task_ep)
    n       = len(ds_ep.samples)
    avg_ibi = (
        np.mean([len(s["ibi_sequence"]) for s in ds_ep.samples])
        if ds_ep.samples else 0.0
    )
    paper_marker = " ← paper default" if epoch_secs == 30 else ""
    print(f"{epoch_secs:<10} {n:<15} {avg_ibi:<20.1f}{paper_marker}")

print(
    "\nObservation: halving epoch duration doubles epoch count "
    "but halves the average IBI count per window."
)

---
## Step 2 — Train a lightweight RNN on the 3-class task

We use PyHealth's built-in **RNN** model as a stand-in for the WatchSleepNet encoder,
applied to the variable-length IBI sequence of each epoch.  
This validates the full data → task → model → evaluation pipeline end-to-end.

In [ ]:
try:
    from pyhealth.datasets import get_dataloader, split_by_patient
    from pyhealth.models import RNN
    from pyhealth.trainer import Trainer

    train_ds, val_ds, test_ds = split_by_patient(ds_3cls, [0.7, 0.15, 0.15])
    train_loader = get_dataloader(train_ds, batch_size=32, shuffle=True)
    val_loader   = get_dataloader(val_ds,   batch_size=32, shuffle=False)
    test_loader  = get_dataloader(test_ds,  batch_size=32, shuffle=False)

    print(f"Split — train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)} epochs")

    model = RNN(
        dataset=ds_3cls,
        feature_keys=["ibi_sequence"],
        label_key="label",
        mode="multiclass",
    )

    trainer = Trainer(model=model, device="cpu")
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=3,
        monitor="accuracy",
    )

    results = trainer.evaluate(test_loader)
    print(f"\nTest results: {results}")

except Exception as exc:
    print(f"[skipped] Model training requires additional dependencies: {exc}")

---
## Cleanup

In [ ]:
if _demo_tmpdir and os.path.isdir(_demo_tmpdir):
    shutil.rmtree(_demo_tmpdir)
    print(f"[demo] Cleaned up {_demo_tmpdir}")

print("Done.")